# Nemotron — PRM + GRPO (RL stage)

This is the **third stage** of the `SFT → PRM → GRPO` pipeline.

- **SFT** (already done): supervised fine-tune on the v15 structured CoTs
  (`## [OBSERVATION] … [ANSWER]` state tags, with `✓`/`✗` step markers).
- **PRM** (this notebook, Part A): train a **Process Reward Model** — a value
  head over the base model that scores each *reasoning step* as good/bad. We
  build step-labelled data from the v15 golden trajectories (positives) and
  programmatic corruptions (negatives), optionally augmented with wrong-answer
  rollouts.
- **GRPO** (this notebook, Part B): **Group-Relative Policy Optimization** of
  the SFT LoRA policy. Reward = verifiable outcome (the competition's own
  `verify()`) + format bonus + (optional) PRM process score. GRPO samples a
  group of completions per prompt, normalises rewards within the group, and
  nudges the policy toward higher-reward trajectories.

**Environment is identical to the SFT notebook**: offline package install
(`--no-deps`, Kaggle libs authoritative), `TRANSFORMERS_OFFLINE=1`,
`WANDB_MODE=offline`, Mamba fast-path + ptxas-blackwell + rmsnorm fixes.
The final adapter stays **eval-server compliant** (plain LoRA, rank ≤ 32,
no `lm_head`/`embed_tokens`, dropout 0).


In [1]:
# ============================================================
# 1. INSTALL DEPENDENCIES  (identical scaffolding to the SFT notebook)
# ============================================================
# Internet is OFF: install from the offline wheels dataset with --no-deps so
# Kaggle's own torch/transformers stay authoritative. GRPO lives in `trl`
# (>=0.9 has GRPOTrainer); everything else matches the SFT recipe.

import subprocess, sys, os, glob
from pathlib import Path

# ============================================================
# STEP 0: ENSURE CLEAN TORCH (no conflicting .so on sys.path)
# ============================================================
# CRITICAL: do NOT extract the NVIDIA metric utility to /tmp or add /tmp to sys.path
# before importing torch — it contains torch C extensions compiled for a different ABI
# that cause: SystemError: ...() method: bad call flags
#
# The SFT notebook works fine without it. We handle ptxas separately in the triton cell.
#
# If you ran the eval notebook in this session, it may have corrupted torch.
# In that case: Runtime → Restart Session, then run THIS notebook first.



TARGET_DIR  = "/kaggle/working/packages"
OFFLINE_DIR = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages"
os.makedirs(TARGET_DIR, exist_ok=True)
if TARGET_DIR not in sys.path:
    sys.path.append(TARGET_DIR)  # APPEND, never insert(0) — Kaggle libs first


def _pip_install(pkgs, *, no_deps=True, no_index=False, find_links=None,
                 path_arg=None, label=None):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--target", TARGET_DIR]
    if no_deps:  cmd.append("--no-deps")
    if no_index: cmd.append("--no-index")
    if find_links: cmd += ["--find-links", find_links]
    if path_arg: cmd.append(path_arg)
    else:        cmd += pkgs
    try:
        subprocess.check_call(cmd)
        if label: print(f"[ok] {label}")
        return True
    except Exception as e:
        if label: print(f"[warn] {label} failed: {e}")
        return False


def _find_wheel(pattern, search_paths):
    for base in search_paths:
        if os.path.isdir(base):
            for f in glob.glob(f"{base}/**/{pattern}", recursive=True):
                return f
    return None


# ---------- nvidia-cutlass (before any CUDA imports) ----------
CUTLASS_PATHS = ["/kaggle/input/datasets/rubyducklove/nvidia-cutlass"]
cutlass_wheel = _find_wheel("nvidia_cutlass-*.whl", CUTLASS_PATHS) \
             or _find_wheel("cutlass-*.whl", CUTLASS_PATHS)
CUTLASS_AVAILABLE = bool(cutlass_wheel) and _pip_install(
    [], path_arg=cutlass_wheel, label=f"nvidia-cutlass <- {cutlass_wheel}")

# ---------- core deps Kaggle is missing (trl carries GRPOTrainer) ----------
PKG_LIST = ["trl", "peft", "datasets", "bitsandbytes", "wandb", "cut-cross-entropy"]
if os.path.isdir(OFFLINE_DIR):
    _pip_install(PKG_LIST, no_index=True, find_links=OFFLINE_DIR,
                 label=f"core deps (offline): {PKG_LIST}")
else:
    _pip_install(PKG_LIST, label=f"core deps (online): {PKG_LIST}")

# ---------- Blackwell Mamba CUDA wheels ----------
WHEEL_PATHS = ["/kaggle/input/datasets/mayukh18/nemotron-packages"]
ccv_wheel  = _find_wheel("causal_conv1d-*.whl", WHEEL_PATHS)
mssm_wheel = _find_wheel("mamba_ssm-*.whl", WHEEL_PATHS)
CAUSAL_CONV1D_AVAILABLE = bool(ccv_wheel) and _pip_install(
    [], path_arg=ccv_wheel, label=f"causal_conv1d <- {ccv_wheel}")
MAMBA_AVAILABLE = bool(mssm_wheel) and _pip_install(
    [], path_arg=mssm_wheel, label=f"mamba_ssm <- {mssm_wheel}")
FAST_PATH_AVAILABLE = MAMBA_AVAILABLE and CAUSAL_CONV1D_AVAILABLE

def _resolve_pth(d):
    for pth in Path(d).glob("*.pth"):
        with pth.open() as fp:
            rel = fp.read().strip()
            p = pth.parent / rel
            if p.exists(): sys.path.append(str(p))
_resolve_pth(TARGET_DIR)



import transformers
TRANSFORMERS_VERSION = tuple(int(x) for x in transformers.__version__.split(".")[:2])
NEW_ENOUGH = TRANSFORMERS_VERSION >= (4, 45)

# Offline env (internet OFF)
os.environ["WANDB_MODE"]          = "offline"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"]       = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

BNB_AVAILABLE = WANDB_AVAILABLE = CCE_AVAILABLE = True
TRL_AVAILABLE = True
for pkg, var in [("bitsandbytes","BNB_AVAILABLE"), ("wandb","WANDB_AVAILABLE"),
                 ("cut_cross_entropy","CCE_AVAILABLE"), ("trl","TRL_AVAILABLE")]:
    try: __import__(pkg)
    except Exception: globals()[var] = False

# Does this trl expose GRPO?
GRPO_AVAILABLE = False
try:
    from trl import GRPOTrainer, GRPOConfig  # noqa
    GRPO_AVAILABLE = True
except Exception as e:
    print(f"[warn] GRPOTrainer not importable from trl: {e}")

print("=" * 60)
print("  Dependency status (RL stage)")
print("=" * 60)
print(f"  transformers     : {transformers.__version__}  {'(OK)' if NEW_ENOUGH else '(TOO OLD)'}")
print(f"  trl / GRPO       : {'YES' if TRL_AVAILABLE else 'NO'} / {'YES' if GRPO_AVAILABLE else 'NO'}")
print(f"  nvidia-cutlass   : {'YES' if CUTLASS_AVAILABLE else 'NO'}")
print(f"  causal_conv1d    : {'YES' if CAUSAL_CONV1D_AVAILABLE else 'NO'}")
print(f"  mamba_ssm        : {'YES' if MAMBA_AVAILABLE else 'NO'}")
print(f"  Mamba fast path  : {'ENABLED' if FAST_PATH_AVAILABLE else 'DISABLED'}")
print(f"  bitsandbytes     : {'YES' if BNB_AVAILABLE else 'NO'}")
print(f"  wandb            : {'YES (offline)' if WANDB_AVAILABLE else 'NO'}")

assert NEW_ENOUGH, f"transformers {transformers.__version__} < 4.45 — Nemotron-H needs >=4.45"
assert GRPO_AVAILABLE, "This trl build has no GRPOTrainer — add a newer trl wheel to the offline dataset."

# Purge Kaggle utility-script mamba_ssm (Mamba3 -> needs cutlass DSL)
_BAD = ("nvidia_utility_script", "nvidia-utility-script")
sys.path[:] = [p for p in sys.path if not any(b in p for b in _BAD)]
for _m in list(sys.modules):
    _f = getattr(sys.modules[_m], "__file__", "") or ""
    if any(b in _f for b in _BAD): del sys.modules[_m]
if TARGET_DIR in sys.path: sys.path.remove(TARGET_DIR)
sys.path.insert(0, TARGET_DIR)
print("[ok] purged utility-script paths; TARGET_DIR first on sys.path")

# ============================================================
# 2. IMPORTS & ENVIRONMENT  (mirrors SFT cell 2, + RL/eval imports)
# ============================================================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import stat, shutil, zipfile, time, json, re, glob, uuid, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, TrainerCallback, BitsAndBytesConfig,
)
from peft import (
    LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training,
)
from trl import GRPOTrainer, GRPOConfig
from tqdm.auto import tqdm

if WANDB_AVAILABLE:
    import wandb

print(f"PyTorch       : {torch.__version__}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"transformers  : {transformers.__version__}")
print(f"W&B           : {'offline' if WANDB_AVAILABLE else 'disabled'}")


[ok] nvidia-cutlass <- /kaggle/input/datasets/rubyducklove/nvidia-cutlass/nvidia_cutlass-4.2.0.0-py3-none-any.whl
[ok] core deps (offline): ['trl', 'peft', 'datasets', 'bitsandbytes', 'wandb', 'cut-cross-entropy']
[ok] causal_conv1d <- /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
[ok] mamba_ssm <- /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
[INFO] Running in WANDB offline mode
  Dependency status (RL stage)
  transformers     : 5.0.0  (OK)
  trl / GRPO       : YES / YES
  nvidia-cutlass   : YES
  causal_conv1d    : YES
  mamba_ssm        : YES
  Mamba fast path  : ENABLED
  bitsandbytes     : YES
  wandb            : YES (offline)
[ok] purged utility-script paths; TARGET_DIR first on sys.path
PyTorch       : 2.10.0+cu128
GPU           : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM          : 95.0 GB
transformers  : 5.0.0
W&B  

In [2]:
# ============================================================
# 2b. WEIGHTS & BIASES — OFFLINE MODE  (same as SFT)
# ============================================================
RUN_HASH         = str(uuid.uuid4())[:8]
DATASET_VERSION  = "v15-PRM-structured-CoT"
NOTEBOOK_VERSION = "nemotron_PRM_GRPO using R32 A32"
print(f"Critical RUN HASH: {RUN_HASH}")
WANDB_PROJECT  = "nemotron-rl-prm-grpo"
WANDB_RUN_NAME = f"{NOTEBOOK_VERSION}-{DATASET_VERSION}-{RUN_HASH}"
WANDB_DIR      = "/kaggle/working"

if WANDB_AVAILABLE:
    wandb.init(
        project=WANDB_PROJECT, name=WANDB_RUN_NAME, dir=WANDB_DIR,
        config={
            "run_hash": RUN_HASH,
            "notebook_version": NOTEBOOK_VERSION,
            "dataset_version": DATASET_VERSION,
            "model": "Nemotron-3-Nano-30B-A3B",
            "stage": "PRM + GRPO (RLVR)",
            "lora_rank": 32, "lora_alpha": 32, "lora_dropout": 0.0,
            "reward": "verify() outcome + format + optional PRM",
        },
        tags=["nemotron", "rl", "grpo", "prm", RUN_HASH, DATASET_VERSION],
    )
    print(f"W&B offline run initialized: {wandb.run.dir}")
else:
    print("W&B not available — metrics to stdout only.")


Critical RUN HASH: 4bb804a1


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: ERROR Unable to load kaggle notebook.
wandb: ERROR Unable to load kaggle notebook.


W&B offline run initialized: /kaggle/working/wandb/offline-run-20260604_030917-hax8vrud/files


In [3]:
# ============================================================
# 3. TRITON FIXES — rmsnorm patch + ptxas-blackwell  (copied from SFT)
# ============================================================
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    var = x.pow(2).mean(-1, keepdim=True)
    y = x * torch.rsqrt(var + eps)
    out = y * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None:    out = out * F.silu(z.float())
    return out.to(dtype)

if not globals().get("FAST_PATH_AVAILABLE", False):
    patched = []
    for name, mod in list(sys.modules.items()):
        nlow = name.lower()
        if not any(k in nlow for k in ("mamba","ssm","selective_scan","rmsnorm")): continue
        if nlow.startswith("transformers"): continue
        try:
            if hasattr(mod, "rmsnorm_fn"):
                mod.rmsnorm_fn = _pure_rmsnorm_fn; patched.append(name)
        except Exception: pass
    print(f"[ok] rmsnorm_fn patched in: {patched}" if patched
          else "[info] no mamba rmsnorm_fn to patch yet")
else:
    print("[info] rmsnorm patch skipped — Mamba CUDA fast path available")

candidates = (
    glob.glob("/kaggle/usr/lib/notebooks/**/ptxas-blackwell", recursive=True)
    + glob.glob("/kaggle/usr/lib/notebooks/**/ptxas", recursive=True)
    + glob.glob("/usr/local/cuda*/bin/ptxas", recursive=True)
    + glob.glob("/usr/local/lib/python*/dist-packages/nvidia/cuda_nvcc/bin/ptxas", recursive=True)
)
src = next((c for c in candidates if "blackwell" in c), None) or (candidates[0] if candidates else None)
if src and os.path.exists(src):
    dst = "/tmp/ptxas-blackwell"
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    for v in ("TRITON_PTXAS_PATH","TRITON_PTXAS_BLACKWELL_PATH","TRITON_PTXAS_BIN","TRITON_PTXAS"):
        os.environ[v] = dst
    try:
        import triton.backends.nvidia.compiler as nv_compiler
        try: nv_compiler.get_ptxas_version.cache_clear()
        except AttributeError: pass
        nv_compiler.get_ptxas_version = lambda arch: "release 12.8"
        from triton import knobs as triton_knobs
        for attr in ("ptxas","ptxas_blackwell"): triton_knobs.nvidia.__dict__.pop(attr, None)
    except Exception as e:
        print(f"[warn] Triton cache clear: {e}")
    print(f"[ok] ptxas binary -> {dst}  (from {src})")
else:
    print("[warn] no ptxas binary found — Mamba Triton kernel may crash")


[info] rmsnorm patch skipped — Mamba CUDA fast path available
[ok] ptxas binary -> /tmp/ptxas-blackwell  (from /usr/local/cuda-12.8/bin/ptxas)


In [4]:
# ============================================================
# 4. EVAL-SERVER UTILITIES  (verbatim from the evaluation notebook)
#    These define the VERIFIABLE REWARD used by GRPO.
# ============================================================
def extract_final_answer(text):
    if text is None: return "NOT_FOUND"
    boxed_starts = list(re.finditer(r"\\boxed\{", text))
    matches = []
    for i, m in enumerate(boxed_starts):
        start = m.end()
        end = boxed_starts[i + 1].start() if i + 1 < len(boxed_starts) else len(text)
        segment = text[start:end]
        last_brace = segment.rfind("}")
        matches.append(segment[:last_brace] if last_brace != -1 else segment)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        return non_empty[-1] if non_empty else matches[-1].strip()
    patterns = [
        r"The final answer is:\s*([^\n]+)", r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:：]\s*([^\n]+)", r"final answer\s*[:：]\s*([^\n]+)",
    ]
    for pattern in patterns:
        m = re.findall(pattern, text, re.IGNORECASE)
        if m: return m[-1].strip()
    m = re.findall(r"-?\d+(?:\.\d+)?", text)
    if m: return m[-1]
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else "NOT_FOUND"

def verify(stored_answer, predicted):
    stored_answer, predicted = str(stored_answer).strip(), str(predicted).strip()
    if re.fullmatch(r"[01]+", stored_answer):
        return predicted.lower() == stored_answer.lower()
    try:
        return math.isclose(float(stored_answer), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()

def categorize_task(row):
    task_id = str(row.get("id", row.get("row_id", ""))).lower()
    prompt  = str(row.get("prompt", row.get("question", ""))).lower()
    category = "Other"
    if "bit" in task_id or "binary" in prompt or "gf2" in task_id: category = "Bit manipulation"
    elif "grav" in task_id or "physics" in task_id: category = "Gravity"
    elif "unit" in task_id or "convert" in prompt: category = "Unit conversion"
    elif "roman" in task_id or "roman" in prompt: category = "Roman numerals"
    elif "cipher" in task_id or "crypto" in task_id: category = "Text ciphers"
    elif "eq" in task_id or "math" in task_id: category = "Equation transforms"
    return category

# Canonical v15 state tags — used for PRM step segmentation and format reward.
STATE_TAGS = ["[OBSERVATION]","[CONSTRAINT]","[ABSTRACTION]","[HYPOTHESIS]",
              "[EVALUATION]","[SELECTION]","[VERIFICATION]","[ANSWER]"]

def split_into_steps(text):
    """Split an assistant CoT into (tag, span_text) steps at '## [' boundaries.
    Returns list of (header, start_char, end_char)."""
    marks = [(m.start(), m.group(1)) for m in re.finditer(r"##\s*(\[[A-Z]+\])", text)]
    steps = []
    for i, (pos, tag) in enumerate(marks):
        end = marks[i + 1][0] if i + 1 < len(marks) else len(text)
        steps.append((tag, pos, end))
    return steps

print("Eval utilities ready: extract_final_answer, verify, categorize_task, split_into_steps")


Eval utilities ready: extract_final_answer, verify, categorize_task, split_into_steps


In [ ]:
# ============================================================
# 5. CONFIG — paths, LoRA contract, PRM + GRPO knobs
# ============================================================
IN_KAGGLE = os.path.exists("/kaggle")

MODEL_PATH   = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
SFT_ADAPTER  = "/kaggle/input/models/manish756/nvidia-adapter/transformers/default/7"

DATA_PATH = Path("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge") \
            if IN_KAGGLE else Path("./data")

V15_FILES = [
    "train_cot_bit_manipulation.jsonl", "train_cot_cipher.jsonl",
    "train_cot_cryptarithm_deduce.jsonl", "train_cot_cryptarithm_guess.jsonl",
    "train_cot_equation_numeric_deduce.jsonl", "train_cot_equation_numeric_guess.jsonl",
    "train_cot_gravity.jsonl", "train_cot_numeral.jsonl", "train_cot_unit_conversion.jsonl",
]
V15_DIR_CANDIDATES = [
    "/kaggle/input/datasets/asharamkanderiwal/nvidia-grpo-prm-dataset/all_categorical_splits_v15",
    str(Path.cwd() / "all_categorical_splits_v15"),
    str(Path.cwd().parent / "all_categorical_splits_v15"),
]

# ---- RESUME FROM CHECKPOINT ----
# Set RESUME_FROM_STEP > 0 to resume GRPO from a saved checkpoint.
# Upload the step_000020/ folder + prm/ folder as a Kaggle dataset first.
RESUME_FROM_STEP     = 0       # 0 = fresh start from SFT (old checkpoints are alpha-64 tainted)
RESUME_ADAPTER_PATH  = "/kaggle/input/datasets/manish756/grpo-resume-ckpt/step_000020"
RESUME_PRM_PATH      = "/kaggle/input/datasets/manish756/grpo-resume-ckpt/prm"
RESUMING = RESUME_FROM_STEP > 0 and os.path.isdir(RESUME_ADAPTER_PATH)

# ---- LoRA contract ----
LORA_RANK    = 32
LORA_ALPHA   = 32          # MUST match the SFT adapter (all SFT runs are R32 A32!)
LORA_DROPOUT = 0.0
MAX_SEQ_LEN  = 8192
MOE_LORA_MODE = "tied"
FORCE_MODE   = "bf16"
MODE         = FORCE_MODE
USE_QLORA    = MODE in ("nf4", "int8")
USE_MAMBA_FAST_PATH = True

# ---- PRM (Part A) ----
PRM_MAX_TRAJ         = 1800
PRM_NEG_PER_POS      = 1
PRM_EPOCHS           = 1
PRM_BATCH            = 1
PRM_GRAD_ACCUM       = 16
PRM_LR               = 1e-4
PRM_MAX_LEN          = 4096
PRM_ADAPTER_NAME     = "prm"
USE_ROLLOUT_NEGATIVES = False

# ---- RFT (Rejection Sampling Fine-tuning) ----
# GRPO can't work on this model: logits are too peaked for within-group
# diversity even at temp=1.5 with forced do_sample. Switch to RFT:
# generate completions, filter correct ones, SFT on them.
TRAIN_GRPO           = False
USE_PRM_REWARD       = False
TRAIN_PRM            = False

RFT_NUM_UNSOLVED     = 80     # unsolved prompts -> generate completions
RFT_NUM_SOLVED       = 200     # solved prompts -> use golden CoTs directly
RFT_COMPLETIONS      = 4       # completions per unsolved prompt
RFT_MAX_PROMPT       = 512
RFT_MAX_COMPLETION   = 384     # shorter = 2.5x faster generation (O(n^2) w/o KV cache)
RFT_TEMPERATURE      = 1.0     # balanced diversity + accuracy
RFT_GEN_HOURS        = 6.0     # time guard for generation phase

RFT_SFT_LR           = 4e-5    # 10x lower than original SFT LR (continuation)
RFT_SFT_EPOCHS       = 2
RFT_SFT_BATCH        = 2
RFT_SFT_GRAD_ACCUM   = 2       # effective batch = 4

# keep these for cells that still reference them
GRPO_TEMPERATURE     = RFT_TEMPERATURE
GRPO_NUM_GENERATIONS = RFT_COMPLETIONS
GRPO_NUM_PROMPTS     = RFT_NUM_UNSOLVED + RFT_NUM_SOLVED
GRPO_MAX_PROMPT_LEN  = RFT_MAX_PROMPT
GRPO_MAX_COMPLETION  = RFT_MAX_COMPLETION
GRPO_USE_VLLM        = False
REWARD_WEIGHTS       = {"correct": 1.0, "format": 0.3, "prm": 0.3}
# ---- Time guard ----
KAGGLE_TIMEOUT_HOURS = 11.5
GRPO_START_TIME      = None

OUTPUT_DIR = "/kaggle/working/grpo_adapter"
PRM_DIR    = "/kaggle/working/prm"
CKPT_DIR   = "/kaggle/working/checkpoints"
for d in (OUTPUT_DIR, PRM_DIR, CKPT_DIR): os.makedirs(d, exist_ok=True)

if USE_MAMBA_FAST_PATH and not FAST_PATH_AVAILABLE:
    print("[warn] fast path wheels missing — disabling")
    USE_MAMBA_FAST_PATH = False

assert LORA_RANK <= 32
assert MAX_SEQ_LEN <= 8192

print("=" * 60)
print("  RL CONFIG — PRM + GRPO")
print("=" * 60)
if RESUMING:
    print(f"  *** RESUMING from step {RESUME_FROM_STEP} ***")
    print(f"  Adapter : {RESUME_ADAPTER_PATH}")
    print(f"  PRM     : {RESUME_PRM_PATH}")
print(f"  LoRA          : r={LORA_RANK}, alpha={LORA_ALPHA}")
print(f"  PRM           : train={TRAIN_PRM}")


print(f"  RFT           : unsolved={RFT_NUM_UNSOLVED}  solved={RFT_NUM_SOLVED}")
print(f"  RFT           : completions/prompt={RFT_COMPLETIONS}  max_completion={RFT_MAX_COMPLETION}")
print(f"  RFT           : gen temp={RFT_TEMPERATURE}  gen time limit={RFT_GEN_HOURS}h")
print(f"  SFT           : LR={RFT_SFT_LR:.1e}  epochs={RFT_SFT_EPOCHS}  batch={RFT_SFT_BATCH*RFT_SFT_GRAD_ACCUM}")
print(f"  Time guard    : auto-save at {KAGGLE_TIMEOUT_HOURS}h")

In [ ]:
# ============================================================
# 6. CALLBACKS — live progress + NaN guard + checkpoint zip + time guard
# ============================================================
class LiveProgressCallback(TrainerCallback):
    def __init__(self): self.pbar=None; self.start=None
    def on_train_begin(self, args, state, control, **kw):
        self.pbar = tqdm(total=state.max_steps, desc="RL", unit="step",
                         dynamic_ncols=True, file=sys.stdout); self.start=time.time()
    def on_step_end(self, args, state, control, **kw):
        if self.pbar is None: return
        el=time.time()-self.start; step=state.global_step
        eta=(el/step)*(state.max_steps-step) if step>0 else 0
        last = state.log_history[-1] if state.log_history else {}
        rwd = last.get("reward", last.get("rewards/correct", None))
        msg = f"reward={rwd:.3f}" if isinstance(rwd,(int,float)) else "reward=..."
        self.pbar.set_postfix_str(f"{msg}  el={el/60:.1f}m eta={eta/60:.1f}m")
        self.pbar.update(1); sys.stdout.flush()
    def on_train_end(self, args, state, control, **kw):
        if self.pbar: self.pbar.close()

class NaNGuardCallback(TrainerCallback):
    def __init__(self, max_consecutive=2): self.max=max_consecutive; self.bad=0
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs: return
        val = logs.get("loss", logs.get("reward", None))
        if val is None: return
        if isinstance(val,(int,float)) and (math.isnan(val) or math.isinf(val)):
            self.bad += 1
            print(f"\n[NaN GUARD] non-finite metric={val} (streak={self.bad})")
            if self.bad >= self.max:
                print("[NaN GUARD] HALTING — diverged."); control.should_training_stop=True
        else:
            self.bad = 0

class CheckpointZipCallback(TrainerCallback):
    """Checkpoint the POLICY adapter only, zip-only (no folder kept), and
    delete the previous zip — Kaggle output is capped at 20 GB."""
    def __init__(self, ckpt_dir, output_dir, every_steps=10):
        self.ckpt_dir=ckpt_dir; self.output_dir=output_dir; self.every=every_steps
        self.prev_zip=None
    def _save(self, state, model_to_save):
        step = state.global_step
        d = os.path.join(self.output_dir, f"step_{step:06d}")
        os.makedirs(d, exist_ok=True)
        # save ONLY the policy adapter — the PRM is already in the input dataset
        try:
            model_to_save.save_pretrained(d, selected_adapters=["default"])
        except TypeError:
            model_to_save.save_pretrained(d)
        # PEFT may nest under default/ — flatten if so
        nested = os.path.join(d, "default")
        if os.path.isdir(nested):
            for fn in os.listdir(nested):
                shutil.move(os.path.join(nested, fn), os.path.join(d, fn))
            os.rmdir(nested)
        cfgp = os.path.join(d, "adapter_config.json")
        if os.path.exists(cfgp):
            cfg = json.load(open(cfgp))
            cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
            json.dump(cfg, open(cfgp, "w"), indent=2)
        zp = os.path.join(self.ckpt_dir, f"grpo_step_{step:06d}_{RUN_HASH}.zip")
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
            for fn in sorted(os.listdir(d)):
                fp=os.path.join(d,fn)
                if os.path.isfile(fp): zf.write(fp, arcname=fn)
        # keep ZIP only — delete the unzipped folder immediately
        shutil.rmtree(d, ignore_errors=True)
        # delete the PREVIOUS zip only after the new one is fully written
        if self.prev_zip and os.path.exists(self.prev_zip):
            try: os.remove(self.prev_zip)
            except Exception as e: print(f"[ckpt] could not remove {self.prev_zip}: {e}")
        self.prev_zip = zp
        free_gb = shutil.disk_usage("/kaggle/working").free/1e9
        print(f"\n[ckpt] {os.path.basename(zp)} ({os.path.getsize(zp)/1e6:.1f} MB)  disk free: {free_gb:.1f} GB")
    def on_step_end(self, args, state, control, model=None, **kw):
        if state.global_step==0 or state.global_step % self.every != 0: return
        self._save(state, model or kw.get("model"))

class DiskGuardCallback(TrainerCallback):
    """Stop training (and let the final save run) if disk gets low."""
    def __init__(self, min_free_gb=3.0):
        self.min_free = min_free_gb
    def on_step_end(self, args, state, control, **kw):
        free_gb = shutil.disk_usage("/kaggle/working").free/1e9
        if free_gb < self.min_free:
            print(f"\n[DISK GUARD] only {free_gb:.1f} GB free — stopping to save adapter")
            control.should_training_stop = True

class TimeGuardCallback(TrainerCallback):
    """Auto-stop before Kaggle timeout so final adapter is saved."""
    def __init__(self, notebook_start_time, max_hours=11.5):
        self.t0 = notebook_start_time; self.max_sec = max_hours * 3600
    def on_step_end(self, args, state, control, model=None, **kw):
        elapsed = time.time() - self.t0
        if elapsed > self.max_sec:
            print(f"\n[TIME GUARD] {elapsed/3600:.1f}h elapsed — stopping to save adapter")
            control.should_training_stop = True

NOTEBOOK_START_TIME = time.time()  # reset at each run

live_cb = LiveProgressCallback(); nan_cb = NaNGuardCallback()
ckpt_cb = CheckpointZipCallback(CKPT_DIR, OUTPUT_DIR, every_steps=10)
time_cb = TimeGuardCallback(NOTEBOOK_START_TIME, max_hours=KAGGLE_TIMEOUT_HOURS)
disk_cb = DiskGuardCallback(min_free_gb=3.0)
print("Callbacks ready: LiveProgress + NaNGuard + CheckpointZip(zip-only, every=10) + TimeGuard + DiskGuard")


In [7]:
# ============================================================
# 7. LOAD BASE MODEL (bf16 + Mamba fast path)  — loaded ONCE, shared by
#    PRM (adapter 'prm') and GRPO (adapter 'default'/policy).
# ============================================================
flash_whl = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
if os.path.exists(flash_whl):
    try:
        subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-index",flash_whl])
        print("[ok] flash_attn installed")
    except Exception as e:
        print(f"[warn] flash_attn skipped: {e}")

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
t_load = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model in bf16 (~3 min)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map={"": 0}, trust_remote_code=True,
    dtype=torch.bfloat16, low_cpu_mem_usage=True, attn_implementation="eager",
)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Enable Mamba CUDA fast path (the big memory saver)
nemotron_mod = None
for _name, _m in list(sys.modules.items()):
    if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
        nemotron_mod = _m; break
if nemotron_mod is not None and USE_MAMBA_FAST_PATH:
    try:
        from causal_conv1d import causal_conv1d_fn
        _x = torch.randn(1,256,32,device="cuda",dtype=torch.bfloat16)
        _w = torch.randn(256,4,device="cuda",dtype=torch.bfloat16)
        causal_conv1d_fn(_x,_w,None,activation="silu")
        import mamba_ssm  # noqa
        nemotron_mod.is_fast_path_available = True
        print("[OK] Mamba FAST PATH ENABLED")
    except Exception as e:
        nemotron_mod.is_fast_path_available = False
        print(f"[warn] fast path check failed -> Python scan: {e}")
elif nemotron_mod is not None:
    nemotron_mod.is_fast_path_available = False

load_min=(time.time()-t_load)/60
print(f"Base loaded in {load_min:.1f} min  |  VRAM {torch.cuda.memory_allocated()/1e9:.1f} GB")


[ok] flash_attn installed
Loading base model in bf16 (~3 min)...


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

[OK] Mamba FAST PATH ENABLED
Base loaded in 7.1 min  |  VRAM 63.2 GB


In [8]:
# ============================================================
# 8. DISCOVER LoRA TARGET MODULES (attention + Mamba + tied MoE)
#    Same selection logic as the SFT notebook — reused for both PRM and policy.
# ============================================================
from collections import Counter
linear_suffixes = Counter()
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        linear_suffixes[name.split(".")[-1]] += 1

ATTENTION_NAMES = ["q_proj","k_proj","v_proj","o_proj","Wqkv","qkv_proj"]
MAMBA_NAMES     = ["in_proj","out_proj","x_proj","dt_proj"]
MLP_GATE_NAMES  = ["gate_proj","gate_up_proj","w1","linear_fc1","fc1"]
MLP_UP_NAMES    = ["up_proj","w3","wi_1"]
MLP_DOWN_NAMES  = ["down_proj","w2","linear_fc2","fc2","wo"]
EXCLUDE = {"lm_head","embed_tokens","shared","router","score","classifier"}

LORA_TARGET_MODULES = []; seen=set()
def add_if_present(names):
    for n in names:
        if n in linear_suffixes and n not in seen and n not in EXCLUDE:
            LORA_TARGET_MODULES.append(n); seen.add(n)
add_if_present(ATTENTION_NAMES); add_if_present(MAMBA_NAMES)
if MOE_LORA_MODE in ("tied","full"):
    add_if_present(MLP_GATE_NAMES); add_if_present(MLP_UP_NAMES); add_if_present(MLP_DOWN_NAMES)
assert LORA_TARGET_MODULES, f"No targets! {dict(linear_suffixes)}"
print(f"LoRA target modules: {LORA_TARGET_MODULES}")

def make_lora_config():
    return LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, target_modules=list(LORA_TARGET_MODULES),
        lora_dropout=LORA_DROPOUT, bias="none", task_type=TaskType.CAUSAL_LM,
    )


LoRA target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj', 'up_proj', 'down_proj']


## Part A — Process Reward Model (PRM)

The PRM is a **scalar value head** over the base model that, at each
`## [STATE]` boundary, predicts whether the reasoning *so far* is on track
(`1.0`) or has gone wrong (`0.0`).

**Training data (outcome→process, Math-Shepherd-lite):**
- **Positives** — the v15 golden trajectories. Every step is correct, so every
  step-boundary gets label `1.0`.
- **Negatives** — programmatic corruptions of the same trajectories:
  (a) the `\boxed{}` answer is swapped to a wrong value → the `[VERIFICATION]`
  and `[ANSWER]` steps become `0.0`; (b) a mid-trajectory step's numbers are
  perturbed → that step onward becomes `0.0`. Optionally, real wrong-answer
  rollouts from the SFT policy (all steps `0.0`) when `USE_ROLLOUT_NEGATIVES`.

The PRM is a LoRA adapter (`'prm'`) on the shared base + a tiny fp32 value head,
so it costs almost no extra VRAM and can later score steps during GRPO.


In [11]:
# ============================================================
# 9. BUILD PRM STEP-LABELLED DATASET (positives + corruptions)
# ============================================================
v15_dir = next((c for c in V15_DIR_CANDIDATES
                if c and os.path.isdir(c)
                and any(os.path.exists(os.path.join(c,f)) for f in V15_FILES)), None)
assert v15_dir, f"v15 dir not found. Searched: {V15_DIR_CANDIDATES}"
print(f"v15 dir: {v15_dir}")

golden = []
for fn in V15_FILES:
    fp = os.path.join(v15_dir, fn)
    if not os.path.exists(fp): continue
    cat = fn.replace("train_cot_","").replace(".jsonl","")
    for line in open(fp):
        if not line.strip(): continue
        rec = json.loads(line)
        asst = rec["messages"][-1]["content"]
        user = rec["messages"][0]["content"]
        golden.append({"category": rec.get("category", cat), "prompt": user, "cot": asst})
random.Random(0).shuffle(golden)
golden = golden[:PRM_MAX_TRAJ]
print(f"Loaded {len(golden)} golden trajectories")

_NUM_RE = re.compile(r"-?\d+")
def _corrupt_answer(cot):
    """Swap the final \boxed{...} to a wrong value; return (new_cot, bad_from_char)."""
    boxes = list(re.finditer(r"\\boxed\{([^}]*)\}", cot))
    if not boxes: return None
    b = boxes[-1]; val = b.group(1)
    nums = _NUM_RE.findall(val)
    if nums:
        n = int(nums[-1]); wrong = str(n + random.choice([-3,-2,-1,1,2,3]) or n+7)
        new_val = val[::-1].replace(nums[-1][::-1], wrong[::-1], 1)[::-1]
    else:
        new_val = (val + "x") if val else "0"
    new_cot = cot[:b.start()] + "\\boxed{" + new_val + "}" + cot[b.end():]
    # bad region starts at the [VERIFICATION] step (the check no longer holds)
    steps = split_into_steps(new_cot)
    bad_from = next((s for (tag,s,e) in steps if tag=="[VERIFICATION]"),
                    (steps[-1][1] if steps else b.start()))
    return new_cot, bad_from

def _corrupt_mid(cot):
    """Perturb a digit inside a mid-trajectory step; bad from that step on."""
    steps = split_into_steps(cot)
    cand = [(tag,s,e) for (tag,s,e) in steps if tag in
            ("[ABSTRACTION]","[HYPOTHESIS]","[EVALUATION]","[SELECTION]")]
    if not cand: return None
    tag,s,e = random.choice(cand)
    seg = cot[s:e]; nums=list(_NUM_RE.finditer(seg))
    if not nums: return None
    m = random.choice(nums); d=m.group()
    repl = str(int(d)+random.choice([-2,-1,1,2]))
    new_seg = seg[:m.start()] + repl + seg[m.end():]
    return cot[:s] + new_seg + cot[e:], s

def char_to_step_labels(cot, bad_from_char):
    """Return list of (end_char_of_step, label) where label=1 before bad_from, else 0."""
    out=[]
    for (tag,s,e) in split_into_steps(cot):
        label = 0.0 if (bad_from_char is not None and s >= bad_from_char) else 1.0
        out.append((e-1, label))   # label sits on the last char of the step
    return out

prm_examples = []  # {prompt, cot, step_labels:[(end_char,label)]}
for g in golden:
    prm_examples.append({**g, "step_labels": char_to_step_labels(g["cot"], None)})  # positive
    made=0
    for fn in (_corrupt_answer, _corrupt_mid):
        if made >= PRM_NEG_PER_POS: break
        res = fn(g["cot"])
        if res is None: continue
        ncot, bad = res
        prm_examples.append({**g, "cot": ncot, "step_labels": char_to_step_labels(ncot, bad)})
        made += 1

random.Random(1).shuffle(prm_examples)
pos = sum(1 for e in prm_examples for (_,l) in e["step_labels"] if l==1.0)
neg = sum(1 for e in prm_examples for (_,l) in e["step_labels"] if l==0.0)
print(f"PRM examples: {len(prm_examples)}  step-labels: +{pos} / -{neg}")


v15 dir: /kaggle/input/datasets/asharamkanderiwal/nvidia-grpo-prm-dataset/all_categorical_splits_v15
Loaded 1800 golden trajectories
PRM examples: 3600  step-labels: +25200 / -3600


In [12]:
# ============================================================
# 9b. (OPTIONAL) mine wrong-answer rollouts as extra PRM negatives
#     Off by default — needs generation. Uses the SFT policy adapter.
# ============================================================
if USE_ROLLOUT_NEGATIVES:
    print("Mining rollout negatives with the SFT policy (HF generation)...")
    # attach SFT adapter temporarily for generation
    _gen = get_peft_model(model, make_lora_config())
    if os.path.isdir(SFT_ADAPTER):
        _gen.load_adapter(SFT_ADAPTER, adapter_name="default", is_trainable=False)
        _gen.set_adapter("default")
    _gen.eval()
    df_tmp = pd.read_csv(DATA_PATH/"train.csv").sample(n=min(200, 10**9), random_state=3)
    added=0
    for row in tqdm(df_tmp.itertuples(index=False), total=len(df_tmp)):
        user = row.prompt + "\nPlease put your final answer inside `\\boxed{}`."
        msgs=[{"role":"user","content":user}]
        try:
            ids = tokenizer.apply_chat_template(msgs, return_tensors="pt",
                    add_generation_prompt=True, enable_thinking=True).to(model.device)
        except Exception:
            ids = tokenizer(user, return_tensors="pt").input_ids.to(model.device)
        with torch.no_grad():
            out = _gen.generate(ids, max_new_tokens=2048, do_sample=True,
                                temperature=0.9, top_p=0.95, pad_token_id=tokenizer.pad_token_id)
        text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
        if not verify(str(row.answer), extract_final_answer(text)):
            prm_examples.append({"category":"rollout","prompt":row.prompt,"cot":text,
                                 "step_labels": [(len(text)-1, 0.0)] +
                                 [(e-1,0.0) for (_,s,e) in split_into_steps(text)]})
            added+=1
    # detach the temporary generation adapter so PRM training starts clean
    model = _gen.unload() if hasattr(_gen,"unload") else model
    print(f"Added {added} rollout negatives; total PRM examples: {len(prm_examples)}")
else:
    print("[info] USE_ROLLOUT_NEGATIVES=False — using golden + corruption labels only")


[info] USE_ROLLOUT_NEGATIVES=False — using golden + corruption labels only


In [ ]:
# ============================================================
# 10. PRM MODEL = base + LoRA('prm') + fp32 value head ; train with BCE
#     over step-boundary hidden states. Only 'prm' LoRA + head are trainable.
#     When RESUMING: load saved PRM adapter + value head (skip training).
# ============================================================
import gc

if TRAIN_PRM:
    # add the PRM LoRA adapter onto the shared base
    if not hasattr(model, "peft_config"):
        model = get_peft_model(model, make_lora_config(), adapter_name=PRM_ADAPTER_NAME)
    else:
        model.add_adapter(PRM_ADAPTER_NAME, make_lora_config())
    model.set_adapter(PRM_ADAPTER_NAME)
    model.enable_input_require_grads()

    hidden_size = model.config.hidden_size if hasattr(model,"config") else \
                  model.base_model.model.config.hidden_size
    value_head = nn.Linear(hidden_size, 1).to(model.device, dtype=torch.float32)
    nn.init.zeros_(value_head.bias); nn.init.normal_(value_head.weight, std=0.02)

    # cast PRM LoRA params to fp32 (numerical hygiene, same as SFT)
    for n,p in model.named_parameters():
        if ".lora_" in n and PRM_ADAPTER_NAME in n:
            p.data = p.data.to(torch.float32)

    def encode_prm(ex):
        """Tokenize prompt+cot, map step end-chars -> token idx, build label/mask."""
        user = ex["prompt"] + "\nPlease put your final answer inside `\\boxed{}`."
        try:
            head = tokenizer.apply_chat_template([{"role":"user","content":user}],
                        tokenize=False, add_generation_prompt=True)
        except Exception:
            head = f"<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n"
        full = head + ex["cot"]
        enc = tokenizer(full, return_offsets_mapping=True, truncation=True,
                        max_length=PRM_MAX_LEN, return_tensors=None)
        ids = enc["input_ids"]; offs = enc["offset_mapping"]
        base = len(head)
        tgt = [-100.0]*len(ids)
        for (end_char, label) in ex["step_labels"]:
            abs_char = base + end_char
            ti = None
            for i,(a,b) in enumerate(offs):
                if a <= abs_char < b or (b!=0 and b-1<=abs_char<=b):
                    ti = i
            if ti is None:
                cand=[i for i,(a,b) in enumerate(offs) if a<=abs_char and b>0]
                ti = cand[-1] if cand else None
            if ti is not None: tgt[ti] = label
        return {"input_ids": ids, "labels": tgt}

    prm_ds = [encode_prm(e) for e in tqdm(prm_examples, desc="encode PRM")]
    prm_ds = [e for e in prm_ds if any(l>=0 for l in e["labels"])]
    print(f"PRM tokenized examples: {len(prm_ds)}")

    def prm_collate(batch):
        maxlen = max(len(b["input_ids"]) for b in batch)
        pad = tokenizer.pad_token_id
        input_ids, attn, labels = [], [], []
        for b in batch:
            n=len(b["input_ids"]); padlen=maxlen-n
            input_ids.append(b["input_ids"]+[pad]*padlen)
            attn.append([1]*n+[0]*padlen)
            labels.append(b["labels"]+[-100.0]*padlen)
        return (torch.tensor(input_ids), torch.tensor(attn),
                torch.tensor(labels, dtype=torch.float32))

    from torch.utils.data import DataLoader
    loader = DataLoader(prm_ds, batch_size=PRM_BATCH, shuffle=True, collate_fn=prm_collate)

    params = [p for n,p in model.named_parameters() if (".lora_" in n and PRM_ADAPTER_NAME in n)]
    params += list(value_head.parameters())
    opt = torch.optim.AdamW(params, lr=PRM_LR, betas=(0.9,0.95), weight_decay=0.0)
    bce = nn.BCEWithLogitsLoss(reduction="none")

    model.train()
    step=0; running=0.0
    for epoch in range(PRM_EPOCHS):
        for bi,(ids,attn,labels) in enumerate(tqdm(loader, desc=f"PRM epoch {epoch}")):
            ids=ids.to(model.device); attn=attn.to(model.device); labels=labels.to(model.device)
            out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True, use_cache=False)
            h = out.hidden_states[-1].float()
            logits = value_head(h).squeeze(-1)
            mask = (labels != -100.0)
            if mask.any():
                loss = (bce(logits[mask], labels[mask].clamp(0,1))).mean() / PRM_GRAD_ACCUM
                loss.backward(); running += loss.item()*PRM_GRAD_ACCUM
            if (bi+1) % PRM_GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step(); opt.zero_grad(); step+=1
                if step % 10 == 0:
                    avg=running/10; running=0.0
                    print(f"  PRM step {step}  loss={avg:.4f}")
                    if WANDB_AVAILABLE: wandb.log({"prm/loss": avg, "prm/step": step})

    # save PRM adapter + value head
    model.save_pretrained(PRM_DIR, selected_adapters=[PRM_ADAPTER_NAME])
    torch.save(value_head.state_dict(), os.path.join(PRM_DIR, "value_head.pt"))
    json.dump({"hidden_size": int(value_head.in_features)},
              open(os.path.join(PRM_DIR,"prm_meta.json"),"w"))
    print(f"[ok] PRM saved to {PRM_DIR}")

    # freeze PRM so GRPO doesn't update it
    for n,p in model.named_parameters():
        if PRM_ADAPTER_NAME in n: p.requires_grad_(False)
    value_head.eval()
    for p in value_head.parameters(): p.requires_grad_(False)

    # ---- CRITICAL: free PRM training artifacts to reclaim ~5-8 GB VRAM ----
    del opt, loader, prm_ds, prm_examples, params, bce
    gc.collect()
    torch.cuda.empty_cache()
    vram_after = torch.cuda.memory_allocated()/1e9
    print(f"[ok] PRM cleanup done — VRAM now {vram_after:.1f} GB")

elif RESUMING and os.path.isdir(RESUME_PRM_PATH):
    # ---- RESUME: load PRM adapter + value head from saved checkpoint ----
    print(f"[RESUME] Loading PRM from {RESUME_PRM_PATH}")

    if not hasattr(model, "peft_config"):
        model = get_peft_model(model, make_lora_config(), adapter_name=PRM_ADAPTER_NAME)
    else:
        model.add_adapter(PRM_ADAPTER_NAME, make_lora_config())

    # PEFT saves named adapters in a subfolder: prm/ contains prm/prm/adapter_model.safetensors
    _prm_adapter_dir = os.path.join(RESUME_PRM_PATH, PRM_ADAPTER_NAME)
    if not os.path.isdir(_prm_adapter_dir):
        _prm_adapter_dir = RESUME_PRM_PATH  # flat layout fallback
    print(f"  adapter dir: {_prm_adapter_dir}")
    model.load_adapter(_prm_adapter_dir, adapter_name=PRM_ADAPTER_NAME, is_trainable=False)
    model.set_adapter(PRM_ADAPTER_NAME)

    # load value head (always at the root level)
    meta_path = os.path.join(RESUME_PRM_PATH, "prm_meta.json")
    if os.path.exists(meta_path):
        prm_meta = json.load(open(meta_path))
        hidden_size = prm_meta["hidden_size"]
    else:
        hidden_size = model.config.hidden_size if hasattr(model,"config") else \
                      model.base_model.model.config.hidden_size

    value_head = nn.Linear(hidden_size, 1).to(model.device, dtype=torch.float32)
    vh_path = os.path.join(RESUME_PRM_PATH, "value_head.pt")
    if os.path.exists(vh_path):
        value_head.load_state_dict(torch.load(vh_path, map_location=model.device))
        print(f"[ok] value_head loaded from {vh_path}")
    else:
        print(f"[warn] value_head.pt not found at {vh_path} — using random init")

    # freeze PRM
    for n,p in model.named_parameters():
        if PRM_ADAPTER_NAME in n: p.requires_grad_(False)
    value_head.eval()
    for p in value_head.parameters(): p.requires_grad_(False)
    print(f"[ok] PRM loaded and frozen — VRAM {torch.cuda.memory_allocated()/1e9:.1f} GB")

else:
    value_head = None
    print("[info] TRAIN_PRM=False — skipping PRM training")

## Part B — GRPO (Group-Relative Policy Optimization)

GRPO samples a **group of `G` completions per prompt** from the policy, scores
each with the reward, computes a group-normalised advantage
`A = (r − mean) / std`, and updates the policy toward the above-average
completions (with a KL leash `beta` to the SFT reference).

**Reward = verifiable outcome + format (+ optional PRM):**
- `reward_correct` — `verify(answer, extract_final_answer(completion))` → `1/0`.
  This is the *exact* competition metric (RLVR).
- `reward_format` — small bonus for a single `\boxed{}` and for using the
  `## [STATE]` tag structure the SFT model already learned.
- `reward_prm` — (optional) mean step score from Part A's PRM. Off by default
  because it adds a second forward pass per completion; flip `USE_PRM_REWARD`.

The policy is the **SFT LoRA adapter continued** (`'default'`), rank 32, same
target modules — so the GRPO output stays a drop-in, eval-server-compliant
adapter.


In [ ]:
# ============================================================
# 11. GRPO DATASET — prompts + ground-truth answers from train.csv
#     NVIDIA RLVR recipe: filter to UNSOLVED prompts only (the 0.86 baseline
#     already gets easy ones right — RL budget should target what it gets wrong).
# ============================================================
df = pd.read_csv(DATA_PATH/"train.csv")
df["category"] = df.apply(categorize_task, axis=1)

# --- Filter to unsolved prompts from the 0.86 baseline ---
EVAL_086_CANDIDATES = [
    "/kaggle/input/datasets/asharamkanderiwal/nvidia-grpo-prm-dataset/evaluation_results_086.csv",
    str(Path.cwd() / "evaluation_results_086.csv"),
    str(Path.cwd().parent / "evaluation_results_086.csv"),
]
eval_086_path = next((p for p in EVAL_086_CANDIDATES if os.path.exists(p)), None)

if eval_086_path:
    eval_df = pd.read_csv(eval_086_path)
    # find the correctness column (could be "correct" or "is_correct")
    _corr_col = "is_correct" if "is_correct" in eval_df.columns else "correct"
    if _corr_col in eval_df.columns and "id" in eval_df.columns:
        # is_correct may be string "True"/"False" or bool
        # ---- FRONTIER MIX, not all-unsolved ----
        # Run cdf768f5 proved 100% unsolved prompts give ZERO learning signal:
        # both samples in every group fail -> identical reward -> advantage 0 ->
        # frac_reward_zero_std=1.0 -> grad 0 for all 53 steps.
        # GRPO needs within-group reward VARIANCE: prompts the baseline solves
        # ~sometimes, so one sample succeeds and the other fails.
        wrong_ids = set(eval_df[eval_df[_corr_col].astype(str).str.strip() == "False"]["id"].astype(str))
        df["_id"] = df["id"].astype(str) if "id" in df.columns else df.index.astype(str)
        unsolved = df[df["_id"].isin(wrong_ids)]
        solved   = df[~df["_id"].isin(wrong_ids)]
        n_uns = min(len(unsolved), GRPO_NUM_PROMPTS // 2)
        n_sol = min(len(solved),   GRPO_NUM_PROMPTS - n_uns)
        df = pd.concat([
            unsolved.sample(n=n_uns, random_state=42),
            solved.sample(n=n_sol, random_state=42),
        ]).drop(columns=["_id"]).sample(frac=1.0, random_state=42)
        print(f"[RLVR] Frontier mix: {n_uns} unsolved + {n_sol} solved prompts")
    else:
        print(f"[warn] Eval CSV missing expected columns — using full dataset")
else:
    print("[info] No 0.86 eval results found — using full dataset (no unsolved filter)")

df = df.sample(n=min(GRPO_NUM_PROMPTS, len(df)), random_state=42).reset_index(drop=True)

def build_prompt(p):
    user = p + "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
    try:
        return tokenizer.apply_chat_template([{"role":"user","content":user}],
                    tokenize=False, add_generation_prompt=True, enable_thinking=True)
    except Exception:
        return user

grpo_ds = Dataset.from_dict({
    "prompt": [build_prompt(p) for p in df["prompt"].tolist()],
    "answer": [str(a) for a in df["answer"].tolist()],
    "category": df["category"].tolist(),
})
print(f"GRPO dataset: {len(grpo_ds)} prompts")
print("Category mix:", dict(pd.Series(grpo_ds['category']).value_counts()))


In [ ]:
# ============================================================
# 12. POLICY = continue the SFT LoRA adapter ('default'), trainable
#     When RESUMING: load the step-N checkpoint instead of SFT adapter.
# ============================================================
if not hasattr(model, "peft_config"):
    model = get_peft_model(model, make_lora_config(), adapter_name="default")
elif "default" not in getattr(model, "peft_config", {}):
    model.add_adapter("default", make_lora_config())

# pick the right adapter to initialise the policy from
if RESUMING:
    _policy_src = RESUME_ADAPTER_PATH
    _policy_label = f"GRPO step-{RESUME_FROM_STEP} checkpoint"
else:
    _policy_src = SFT_ADAPTER
    _policy_label = "SFT adapter"

# ---- GUARD: PEFT load_adapter keeps OUR config's alpha, only loads weights.
# If the source adapter was trained at a different alpha, serving scale is wrong
# (this exact bug caused a 0.00 LB score). Hard-fail on mismatch.
_src_cfg_p = os.path.join(_policy_src, "adapter_config.json")
if os.path.exists(_src_cfg_p):
    _src_cfg = json.load(open(_src_cfg_p))
    assert _src_cfg.get("lora_alpha") == LORA_ALPHA, (
        f"ALPHA MISMATCH: {_policy_label} was saved with lora_alpha="
        f"{_src_cfg.get('lora_alpha')} but this notebook uses LORA_ALPHA={LORA_ALPHA}. "
        f"Set LORA_ALPHA={_src_cfg.get('lora_alpha')} in the config cell.")
    assert _src_cfg.get("r") == LORA_RANK, (
        f"RANK MISMATCH: {_src_cfg.get('r')} vs {LORA_RANK}")

if os.path.isdir(_policy_src) and os.path.exists(os.path.join(_policy_src, "adapter_model.safetensors")):
    model.load_adapter(_policy_src, adapter_name="default", is_trainable=True)
    print(f"[ok] policy initialised from {_policy_label}: {_policy_src}")
    for n,p in model.named_parameters():
        if "lora_B" in n and "default" in n:
            assert p.detach().abs().mean().item() > 1e-7, f"{_policy_label} load failed (lora_B=0)"
            break
else:
    print(f"[warn] {_policy_label} not found at {_policy_src} — policy starts from fresh LoRA")

model.set_adapter("default")
model.enable_input_require_grads()

# fp32 cast on the trainable policy LoRA
n_fp32=0
for n,p in model.named_parameters():
    if ".lora_" in n and "default" in n:
        p.data = p.data.to(torch.float32); p.requires_grad_(True); n_fp32+=1
    elif PRM_ADAPTER_NAME in n:
        p.requires_grad_(False)
print(f"[ok] policy LoRA params (fp32, trainable): {n_fp32}")
model.print_trainable_parameters()

In [ ]:
# ============================================================
# 13. REWARD FUNCTIONS  (TRL passes dataset columns as kwargs)
# ============================================================
def _completion_text(c):
    # TRL gives either a string or a chat list [{"role","content"}]
    if isinstance(c, list): return c[-1]["content"]
    return c

def reward_correct(prompts, completions, answer, **kw):
    out=[]
    for comp, ans in zip(completions, answer):
        pred = extract_final_answer(_completion_text(comp))
        out.append(1.0 if verify(ans, pred) else 0.0)
    return out

def reward_format(prompts, completions, **kw):
    out=[]
    for comp in completions:
        t = _completion_text(comp); r = 0.0
        nb = len(re.findall(r"\\boxed\{", t))
        if nb == 1: r += 0.6
        elif nb > 1: r += 0.1
        tags = sum(1 for tg in STATE_TAGS if ("## "+tg) in t or tg in t)
        r += 0.4 * min(tags, 4) / 4.0      # reward using the learned state structure
        out.append(r)
    return out

# ---- optional PRM process reward (uses the frozen 'prm' adapter + value head) ----
@torch.no_grad()
def _prm_score_text(prompt_text, completion_text):
    model.set_adapter(PRM_ADAPTER_NAME)
    try:
        full = prompt_text + completion_text
        enc = tokenizer(full, return_offsets_mapping=True, truncation=True,
                        max_length=PRM_MAX_LEN, return_tensors=None)
        ids = torch.tensor([enc["input_ids"]]).to(model.device)
        attn= torch.ones_like(ids)
        out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True, use_cache=False)
        h = out.hidden_states[-1].float()
        logits = value_head(h).squeeze(-1)[0]
        # score at each step boundary inside the completion
        base = len(prompt_text); offs = enc["offset_mapping"]; scores=[]
        for (tag,s,e) in split_into_steps(completion_text):
            abs_char = base + (e-1)
            cand=[i for i,(a,b) in enumerate(offs) if a<=abs_char and b>0]
            if cand: scores.append(torch.sigmoid(logits[cand[-1]]).item())
        return float(np.mean(scores)) if scores else 0.5
    finally:
        model.set_adapter("default")

def reward_prm(prompts, completions, **kw):
    return [_prm_score_text(p, _completion_text(c)) for p,c in zip(prompts, completions)]

def reward_length(prompts, completions, **kw):
    """Continuous reward inversely proportional to completion length.
    Even near-identical samples have slightly different token counts,
    so this provides within-group variance when binary correctness doesn't."""
    out = []
    for comp in completions:
        t = _completion_text(comp)
        ratio = min(len(t) / 2048.0, 1.0)
        out.append(1.0 - ratio)
    return out

REWARD_FUNCS = [reward_correct, reward_format, reward_length]
WEIGHTS = [REWARD_WEIGHTS["correct"], REWARD_WEIGHTS["format"], REWARD_WEIGHTS.get("length", 0.1)]
# PRM is available if we trained it OR loaded it from checkpoint
_prm_available = value_head is not None
if USE_PRM_REWARD and _prm_available:
    REWARD_FUNCS.append(reward_prm); WEIGHTS.append(REWARD_WEIGHTS["prm"])
    print("[info] PRM process reward ENABLED")
print(f"Reward funcs: {[f.__name__ for f in REWARD_FUNCS]}  weights={WEIGHTS}")

In [ ]:
# ============================================================
# 14. RFT — REJECTION SAMPLING FINE-TUNING
# ============================================================
import gc, time
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ---- lm_head memory patch ----
_nemotron_model = None
_inner = model
for _attr in ("model", "base_model", "model"):
    _inner = getattr(_inner, _attr, _inner)
for _attr in ("model", "base_model"):
    if hasattr(_inner, "lm_head"):
        _nemotron_model = _inner
        break
    _inner = getattr(_inner, _attr, _inner)
if hasattr(_inner, "lm_head"):
    _nemotron_model = _inner

if _nemotron_model is not None:
    _orig_forward = _nemotron_model.__class__.forward
    def _patched_forward(self, input_ids=None, inputs_embeds=None, position_ids=None,
                         cache_params=None, labels=None, output_attentions=None,
                         output_hidden_states=None, return_dict=None, use_cache=None,
                         cache_position=None, attention_mask=None, **kwargs):
        if labels is None and not output_hidden_states and not self.training:
            from transformers.modeling_outputs import CausalLMOutputWithPast
            backbone = self.backbone if hasattr(self, 'backbone') else self.model
            outputs = backbone(
                input_ids=input_ids, inputs_embeds=inputs_embeds,
                cache_params=cache_params, use_cache=use_cache,
                output_hidden_states=False, return_dict=True,
                cache_position=cache_position, attention_mask=attention_mask,
                position_ids=position_ids, **kwargs)
            hidden_states = outputs.last_hidden_state
            logits = self.lm_head(hidden_states[:, -1:, :].to(self.lm_head.weight.dtype)).float()
            return CausalLMOutputWithPast(
                loss=None, logits=logits,
                past_key_values=getattr(outputs, 'past_key_values', None),
                hidden_states=None, attentions=None)
        return _orig_forward(self, input_ids=input_ids, inputs_embeds=inputs_embeds,
                             position_ids=position_ids, cache_params=cache_params,
                             labels=labels, output_attentions=output_attentions,
                             output_hidden_states=output_hidden_states,
                             return_dict=return_dict, use_cache=use_cache,
                             cache_position=cache_position, attention_mask=attention_mask,
                             **kwargs)
    _nemotron_model.__class__.forward = _patched_forward
    print(f"[ok] lm_head patch on {_nemotron_model.__class__.__name__}")

gc.collect(); torch.cuda.empty_cache()

# ================================================================
# PHASE 0: Prepare prompts + golden CoTs
# ================================================================
print("=" * 60)
print("  RFT — Phase 0: Prepare data")
print("=" * 60)

train_df = pd.read_csv(DATA_PATH / "train.csv")
train_df["_id"] = train_df["id"].astype(str)

# Identify solved/unsolved
eval_paths = sorted(DATA_PATH.glob("nvidia_llm_eval*.csv"))
if not eval_paths:
    # Fallback: check known Kaggle dataset location
    _fallback = Path("/kaggle/input/failed-puzzles-dataset/evaluation_results_086.csv")
    if _fallback.exists():
        eval_paths = [_fallback]
if eval_paths:
    eval_df = pd.read_csv(eval_paths[0])
    _cc = [c for c in eval_df.columns if "correct" in c.lower()]
    if _cc:
        wrong_ids = set(eval_df[eval_df[_cc[0]] == False]["id"].astype(str))
    else:
        wrong_ids = set(eval_df["id"].astype(str))
    print(f"  Eval: {len(wrong_ids)} wrong out of {len(eval_df)}")
else:
    wrong_ids = set()
    print("  [warn] No eval CSV — will generate for all prompts")

unsolved = train_df[train_df["_id"].isin(wrong_ids)] if wrong_ids else train_df
solved   = train_df[~train_df["_id"].isin(wrong_ids)] if wrong_ids else pd.DataFrame()

# Load golden CoTs from v15
v15_dir = None
for d in V15_DIR_CANDIDATES:
    if os.path.isdir(d):
        v15_dir = d
        break

golden_cots = {}
golden_cots_by_prompt = {}  # keyed by prompt text (v15 has no id column)
if v15_dir:
    print(f"  v15 dir: {v15_dir}")
    for f in V15_FILES:
        fp = os.path.join(v15_dir, f)
        if not os.path.exists(fp):
            continue
        _df = pd.read_json(fp, lines=True)
        print(f"    {f}: columns={list(_df.columns)}, rows={len(_df)}")
        # v15 data is chat-format: columns=['category', 'messages']
        # messages is a list of [{"role":"user","content":...}, {"role":"assistant","content":...}]
        id_col = next((c for c in ["id", "_id", "row_id"] if c in _df.columns), None)
        added_from_file = 0
        if "messages" in _df.columns:
            for _, row in _df.iterrows():
                msgs = row["messages"]
                if isinstance(msgs, str):
                    try: msgs = json.loads(msgs)
                    except: continue
                if not isinstance(msgs, list): continue
                # Extract assistant response and user prompt
                assistant_text = ""
                user_text = ""
                for m in msgs:
                    if isinstance(m, dict):
                        if m.get("role") == "assistant":
                            assistant_text = m.get("content", "")
                        elif m.get("role") == "user":
                            user_text = m.get("content", "")
                if len(assistant_text) > 20:
                    # Key by prompt text (no id column in v15)
                    prompt_key = user_text.replace("\nPlease put your final answer inside `\\boxed{}`.", "").strip()
                    golden_cots_by_prompt[prompt_key] = assistant_text
                    if id_col:
                        golden_cots[str(row[id_col])] = assistant_text
                    added_from_file += 1
            print(f"      -> messages column, extracted {added_from_file} CoTs")
        else:
            cot_col = next((c for c in _df.columns if c in
                            ["cot", "chain_of_thought", "response", "completion",
                             "output", "text", "reasoning", "solution"]), None)
            if cot_col and id_col:
                for _, row in _df.iterrows():
                    cot_text = str(row[cot_col])
                    if len(cot_text) > 20:
                        golden_cots[str(row[id_col])] = cot_text
                        added_from_file += 1
                print(f"      -> {cot_col} column, {added_from_file} entries")
            else:
                print(f"      -> no usable column found")
else:
    print("  [warn] v15 dir not found at any candidate path")
    for d in V15_DIR_CANDIDATES:
        print(f"    tried: {d} -> exists={os.path.isdir(d)}")

print(f"  Golden CoTs total: {len(golden_cots)}")

# Sample prompts
n_uns = min(len(unsolved), RFT_NUM_UNSOLVED)
n_sol = min(len(solved),   RFT_NUM_SOLVED)
unsolved_sample = unsolved.sample(n=n_uns, random_state=42) if n_uns > 0 else pd.DataFrame()
solved_sample   = solved.sample(n=n_sol, random_state=42) if n_sol > 0 else pd.DataFrame()

# Combine: generate for BOTH solved and unsolved
all_prompts = pd.concat([solved_sample, unsolved_sample]).sample(frac=1, random_state=42)
print(f"  Generation prompts: {len(all_prompts)} ({n_sol} solved + {n_uns} unsolved)")

# ================================================================
# PHASE 1: Generate completions for ALL prompts
# ================================================================
print("\n" + "=" * 60)
print("  RFT — Phase 1: Generate completions")
print("=" * 60)
print(f"  {len(all_prompts)} prompts x {RFT_COMPLETIONS} @ temp={RFT_TEMPERATURE}")
print(f"  max_completion={RFT_MAX_COMPLETION}  time limit={RFT_GEN_HOURS}h\n")

model.eval()
rft_correct = []
stats = {"generated": 0, "correct": 0, "prompts_with_hit": 0, "skipped": 0}
t0 = time.time()

for idx, (_, row) in enumerate(all_prompts.iterrows()):
    elapsed_h = (time.time() - t0) / 3600
    if elapsed_h > RFT_GEN_HOURS:
        print(f"\n[TIME GUARD] {RFT_GEN_HOURS}h reached at prompt {idx}/{len(all_prompts)}")
        break

    prompt_text = str(row["prompt"]) + "\nPlease put your final answer inside `\\boxed{}`."
    answer = str(row["answer"])

    msgs = [{"role": "user", "content": prompt_text}]
    try:
        input_ids = tokenizer.apply_chat_template(
            msgs, return_tensors="pt", add_generation_prompt=True,
            enable_thinking=True).to(model.device)
    except Exception:
        input_ids = tokenizer(prompt_text, return_tensors="pt").input_ids.to(model.device)

    if input_ids.shape[1] > RFT_MAX_PROMPT:
        stats["skipped"] += 1
        continue

    batch_ids = input_ids.expand(RFT_COMPLETIONS, -1)
    try:
        with torch.no_grad():
            outputs = model.generate(
                batch_ids,
                max_new_tokens=RFT_MAX_COMPLETION,
                do_sample=True,
                temperature=RFT_TEMPERATURE,
                top_k=50,
                top_p=0.95,
                pad_token_id=tokenizer.pad_token_id,
            )
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache(); gc.collect()
            stats["skipped"] += 1
            continue
        raise

    prompt_hit = False
    for j in range(outputs.shape[0]):
        text = tokenizer.decode(outputs[j][input_ids.shape[1]:], skip_special_tokens=True)
        stats["generated"] += 1
        pred = extract_final_answer(text)
        if verify(answer, pred):
            rft_correct.append({
                "prompt": str(row["prompt"]),
                "completion": text,
                "source": "rft",
            })
            stats["correct"] += 1
            prompt_hit = True

    if prompt_hit:
        stats["prompts_with_hit"] += 1

    if (idx + 1) % 20 == 0:
        pct = stats["correct"] / max(stats["generated"], 1) * 100
        print(f"  [{idx+1}/{len(all_prompts)}] correct={stats['correct']}/{stats['generated']} "
              f"({pct:.1f}%)  hits={stats['prompts_with_hit']}  "
              f"elapsed={elapsed_h:.1f}h")

    torch.cuda.empty_cache()

gen_h = (time.time() - t0) / 3600
print(f"\nGeneration done: {gen_h:.1f}h")
print(f"  Total completions: {stats['generated']}")
print(f"  Correct: {stats['correct']} ({stats['correct']/max(stats['generated'],1)*100:.1f}%)")
print(f"  Prompts with hit: {stats['prompts_with_hit']}")
print(f"  Skipped (OOM/long): {stats['skipped']}")

# ================================================================
# PHASE 2: Build SFT dataset
# ================================================================
print("\n" + "=" * 60)
print("  RFT — Phase 2: Build SFT dataset")
print("=" * 60)

sft_examples = list(rft_correct)

# Add golden CoTs for prompts not covered by generation
gen_prompt_set = set(ex["prompt"] for ex in rft_correct)
golden_added = 0
for _, row in train_df.iterrows():
    if golden_added >= 500:
        break
    pid = str(row.get("id", ""))
    prompt = str(row["prompt"])
    cot = golden_cots.get(pid) or golden_cots_by_prompt.get(prompt.strip())
    if cot and prompt not in gen_prompt_set:
        sft_examples.append({
            "prompt": prompt,
            "completion": cot,
            "source": "golden",
        })
        golden_added += 1

# Deduplicate: keep first occurrence per prompt
seen = set()
deduped = []
for ex in sft_examples:
    key = ex["prompt"][:200]
    if key not in seen:
        seen.add(key)
        deduped.append(ex)
sft_examples = deduped

print(f"  RFT-generated: {stats['correct']}")
print(f"  Golden CoTs:   {golden_added}")
print(f"  After dedup:   {len(sft_examples)}")

if len(sft_examples) < 5:
    print("[WARN] Very few examples. Saving base adapter as fallback.")
    try:
        model.save_pretrained(OUTPUT_DIR, selected_adapters=["default"])
    except TypeError:
        model.save_pretrained(OUTPUT_DIR)
    cfgp = os.path.join(OUTPUT_DIR, "adapter_config.json")
    if os.path.exists(cfgp):
        cfg = json.load(open(cfgp))
        cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
        json.dump(cfg, open(cfgp, "w"), indent=2)
    print("[ok] Saved untouched SFT adapter as fallback")
else:
    random.shuffle(sft_examples)

    # Tokenize
    print("Tokenizing...")
    tokenized = []
    for ex in sft_examples:
        p = ex["prompt"] + "\nPlease put your final answer inside `\\boxed{}`."
        c = ex["completion"]
        msgs_full   = [{"role": "user", "content": p}, {"role": "assistant", "content": c}]
        msgs_prompt = [{"role": "user", "content": p}]
        try:
            full_ids   = tokenizer.apply_chat_template(msgs_full, tokenize=True, enable_thinking=True)
            prompt_ids = tokenizer.apply_chat_template(msgs_prompt, tokenize=True,
                                                        add_generation_prompt=True, enable_thinking=True)
        except Exception:
            full_ids   = tokenizer.apply_chat_template(msgs_full, tokenize=True)
            prompt_ids = tokenizer.apply_chat_template(msgs_prompt, tokenize=True, add_generation_prompt=True)
        max_len = min(MAX_SEQ_LEN, 4096)
        if len(full_ids) > max_len:
            full_ids = full_ids[:max_len]
        plen = min(len(prompt_ids), len(full_ids))
        labels = [-100] * plen + full_ids[plen:]
        tokenized.append({"input_ids": full_ids, "labels": labels})

    avg_len = np.mean([len(t["input_ids"]) for t in tokenized])
    print(f"  {len(tokenized)} examples, avg {avg_len:.0f} tokens")

    from datasets import Dataset as HFDataset
    sft_ds = HFDataset.from_dict({
        "input_ids": [t["input_ids"] for t in tokenized],
        "labels":    [t["labels"]    for t in tokenized],
    })

    # ================================================================
    # PHASE 3: SFT training
    # ================================================================
    print("\n" + "=" * 60)
    print("  RFT — Phase 3: SFT training")
    print("=" * 60)

    model.train()
    gc.collect(); torch.cuda.empty_cache()

    collator = DataCollatorForSeq2Seq(tokenizer, padding=True, pad_to_multiple_of=8)

    sft_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=RFT_SFT_BATCH,
        gradient_accumulation_steps=RFT_SFT_GRAD_ACCUM,
        learning_rate=RFT_SFT_LR,
        num_train_epochs=RFT_SFT_EPOCHS,
        bf16=True,
        logging_steps=5,
        save_strategy="no",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        max_grad_norm=1.0,
        report_to="wandb" if WANDB_AVAILABLE else "none",
        run_name=(WANDB_RUN_NAME + "_rft_sft") if WANDB_AVAILABLE else None,
        remove_unused_columns=False,
        dataloader_pin_memory=False,
    )

    sft_trainer = Trainer(
        model=model,
        args=sft_args,
        train_dataset=sft_ds,
        data_collator=collator,
    )

    print(f"  Examples: {len(sft_ds)}  Epochs: {RFT_SFT_EPOCHS}")
    print(f"  Batch: {RFT_SFT_BATCH}x{RFT_SFT_GRAD_ACCUM}={RFT_SFT_BATCH*RFT_SFT_GRAD_ACCUM}")
    print(f"  LR: {RFT_SFT_LR:.1e}  Scheduler: cosine\n")

    t_sft = time.time()
    sft_trainer.train()
    sft_h = (time.time() - t_sft) / 3600
    print(f"\nSFT done: {sft_h:.1f}h, {sft_trainer.state.global_step} steps")

    # Save
    print("\nSaving final adapter...")
    try:
        model.save_pretrained(OUTPUT_DIR, selected_adapters=["default"])
    except TypeError:
        model.save_pretrained(OUTPUT_DIR)
    cfgp = os.path.join(OUTPUT_DIR, "adapter_config.json")
    if os.path.exists(cfgp):
        cfg = json.load(open(cfgp))
        cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
        json.dump(cfg, open(cfgp, "w"), indent=2)
    print(f"[ok] Final adapter saved to {OUTPUT_DIR}")

peak = torch.cuda.max_memory_allocated() / 1e9
total_h = (time.time() - t0) / 3600
print(f"Peak VRAM: {peak:.2f} GB")
print(f"Total time: {total_h:.1f}h")


In [ ]:
# ============================================================
# 15. ZIP + COMPLIANCE CHECK
# ============================================================
cfgp = os.path.join(OUTPUT_DIR, "adapter_config.json")
cfg = json.load(open(cfgp))
cfg["_custom_run_hash"]  = RUN_HASH
cfg["_dataset_version"]  = DATASET_VERSION
cfg["_notebook_version"] = NOTEBOOK_VERSION
cfg["_stage"]            = "RFT"
cfg["_grpo_steps"]       = sft_trainer.state.global_step if "sft_trainer" in dir() else 0
json.dump(cfg, open(cfgp,"w"), indent=2)

print(f"base_model_name_or_path -> {cfg['base_model_name_or_path']}")
print(f"peft_type -> {cfg.get('peft_type')}  r/alpha -> {cfg.get('r')}/{cfg.get('lora_alpha')}")

ZIP_PATH = f"/kaggle/working/grpo_adapter_final_{RUN_HASH}.zip"
if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)
contents=[]
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(OUTPUT_DIR)):
        fp=os.path.join(OUTPUT_DIR, fn)
        if os.path.isfile(fp): zf.write(fp, arcname=fn); contents.append(fn)
print(f"\nFINAL adapter.zip ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB): {contents}")

targets = cfg.get("target_modules", [])
checks = [
    ("base_model = metric/...", cfg.get("base_model_name_or_path")=="metric/nemotron-3-nano-30b-a3b-bf16"),
    ("peft_type = LORA",        cfg.get("peft_type")=="LORA"),
    ("rank <= 32",              cfg.get("r",99)<=32),
    ("dropout = 0",             cfg.get("lora_dropout",-1)==0.0),
    ("lm_head NOT in targets",  "lm_head" not in targets),
    ("embed_tokens NOT in tgt", "embed_tokens" not in targets),
    ("adapter_config.json",     "adapter_config.json" in contents),
    ("adapter_model.safetensors", "adapter_model.safetensors" in contents),
]
all_ok=True
print("\nCompliance:")
for name,ok in checks:
    all_ok = all_ok and ok
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
print("\n  ✓ eval-server compliant" if all_ok else "\n  ✗ FIX FAILURES before submitting")


In [ ]:
# ============================================================
# 16. W&B FINISH + LOG ARCHIVE
# ============================================================
if WANDB_AVAILABLE and wandb.run is not None:
    wandb.log({"final/grpo_zip_mb": os.path.getsize(ZIP_PATH)/1e6,
               "final/peak_vram_gb": peak})
    wandb.finish()
    wandb_dir = os.path.join(WANDB_DIR, "wandb")
    if os.path.exists(wandb_dir):
        wz = f"/kaggle/working/wandb_logs_{RUN_HASH}.zip"
        with zipfile.ZipFile(wz, "w", zipfile.ZIP_DEFLATED) as zf:
            for root,_,files in os.walk(wandb_dir):
                for fn in files:
                    fp=os.path.join(root,fn)
                    zf.write(fp, arcname=os.path.relpath(fp, WANDB_DIR))
        print(f"W&B logs zipped: {wz} ({os.path.getsize(wz)/1e6:.1f} MB)")

print("\n" + "="*60)
print("  RL STAGE COMPLETE")
print("="*60)
print(f"  PRM adapter+head : {PRM_DIR}")
print(f"  GRPO adapter zip : {ZIP_PATH}")
print("  -> submit the GRPO adapter, or compare vs SFT on the eval notebook.")
